In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv pillow
import os
import io
import base64
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from config import config
from database import DatabaseManager
from PIL import Image
import ollama
import openai

app = FastAPI()

# 모든 Origin, Method, Header 허용 (가이드 준수)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

dbManager = DatabaseManager()

def analyzeWithGemma(imageBytes, userQuestion):
    """ Ollama gemma4:e2b 모델을 사용한 이미지 분석 """
    try:
        # config 객체에서 모델 이름 참조
        modelName = config.ollamaModel
        response = ollama.generate(
            model=modelName,
            prompt=userQuestion,
            images=[imageBytes]
        )
        return response['response']
    except Exception as e:
        raise e

def analyzeWithGpt(imageBytes, userQuestion):
    """ OpenAI GPT-4o 모델을 사용한 이미지 분석 """
    try:
        client = openai.OpenAI(api_key=config.openaiApiKey)
        # 이미지를 base64로 인코딩
        base64Image = base64.b64encode(imageBytes).decode('utf-8')
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": userQuestion},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64Image}"}}
                    ],
                }
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        raise e

@app.post("/analyze")
async def analyzeImage(image: UploadFile = File(...), question: str = Form(...)):
    """ 이미지 업로드 및 모델 기반 질문 답변 API """
    try:
        # 이미지 데이터 비동기 읽기
        imageData = await image.read()
        
        # 1. 파일 시스템에 이미지 저장
        saveDir = "dataset"
        fileName = image.filename
        savePath = os.path.join(saveDir, fileName)
        
        with open(savePath, "wb") as f:
            f.write(imageData)
            
        # 2. config 객체를 통한 모델 선택 및 분석
        useModel = config.useModel
        resultText = ""
        
        if useModel == "OLLAMA":
            resultText = analyzeWithGemma(imageData, question)
        elif useModel == "GPT":
            resultText = analyzeWithGpt(imageData, question)
        else:
            return {"success": False, "message": "잘못된 모델 설정입니다."}
            
        # 3. 분석 결과 DB 저장
        insertQuery = "INSERT INTO image_analysis (fileName, question, answer) VALUES (%s, %s, %s)"
        dbManager.executeQuery(insertQuery, (fileName, question, resultText))
        
        return {"success": True, "result": resultText}
        
    except Exception as e:
        return {"success": False, "message": str(e)}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)